# Advanced KPIs · staging import

**Optional database write step.** First run SQL 11 to create the staging table. Input: advanced_analysis.pkl. After this import, run SQL 12 and 13, then the validation notebook. This step stops on a nonempty staging table.

Run cells from top to bottom in a **fresh kernel**. The setup cell declares inputs explicitly. Code comments and explanations are in English. The original source cell numbers are recorded in notebook metadata and in `docs/cell_mapping.csv`.

## Setup

Locate the repository and import shared utilities. This does not change the database.

In [ ]:
from pathlib import Path
import sys
import json
from urllib.request import urlopen
import numpy as np
import pandas as pd
from IPython.display import display

# Find the repository from any notebook subfolder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "scouting" / "config.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the extracted repository.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from scouting.config import PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, REPORTS_DIR, IMPECT_BASE_URL, ITERATION_ID
from scouting.data_access import load_or_download_json, load_metadata_context, metadata
from scouting.checkpoints import save_context, load_context, BASE_VARIABLES, ADVANCED_VARIABLES
print("Project:", PROJECT_ROOT)

globals().update(load_context("advanced_analysis"))
import pyodbc
from scouting.database import connection_string as build_connection_string, require_load_enabled
connection_string = build_connection_string()
RUN_DATABASE_LOAD = False


## Load and verify the advanced payload

Insert only into an empty advanced staging table, read back all keys and values, compare NULLs, and commit only on full agreement.

In [ ]:
require_load_enabled(RUN_DATABASE_LOAD)

import math
import numpy as np

# Prepare SQL-compatible values.
def optional_kpi_value(value):
    if pd.isna(value):
        return None

    number = float(value)

    if not math.isfinite(number):
        raise ValueError("Invalid numeric KPI value.")

    return number


advanced_sql_columns = [
    "match_id",
    "team_id",
    "player_id",
    "position_name",
    "kpi_id",
    "player_aggregate_value",
    "event_export_value",
]

advanced_sql_rows = [
    (
        int(row.match_id),
        int(row.team_id),
        int(row.player_id),
        str(row.position_name),
        int(row.kpi_id),
        optional_kpi_value(row.player_aggregate_value),
        optional_kpi_value(row.event_export_value),
    )
    for row in advanced_kpis_to_load.itertuples(index=False)
]

connection = pyodbc.connect(
    connection_string,
    timeout=10,
    autocommit=False,
)

try:
    cursor = connection.cursor()

    # Prevent inserting the same dataset twice.
    cursor.execute("""
        SELECT COUNT_BIG(*)
        FROM staging.player_match_position_kpi;
    """)

    if cursor.fetchone()[0] != 0:
        raise RuntimeError(
            "The advanced KPI table already contains data. "
            "Import stopped to prevent duplicates."
        )

    cursor.fast_executemany = True

    cursor.executemany("""
        INSERT INTO staging.player_match_position_kpi (
            match_id,
            team_id,
            player_id,
            position_name,
            kpi_id,
            player_aggregate_value,
            event_export_value
        )
        VALUES (?, ?, ?, ?, ?, ?, ?);
    """, advanced_sql_rows)

    # Read back the inserted records before committing.
    cursor.execute("""
        SELECT
            match_id,
            team_id,
            player_id,
            position_name,
            kpi_id,
            player_aggregate_value,
            event_export_value
        FROM staging.player_match_position_kpi;
    """)

    sql_advanced_check = pd.DataFrame.from_records(
        [tuple(row) for row in cursor.fetchall()],
        columns=advanced_sql_columns,
    )

    if len(sql_advanced_check) != len(advanced_kpis_to_load):
        raise ValueError("SQL and Python row counts differ.")

    comparison = advanced_kpis_to_load.merge(
        sql_advanced_check,
        on=advanced_sql_keys,
        how="outer",
        suffixes=("_python", "_sql"),
        validate="one_to_one",
        indicator=True,
    )

    if not comparison["_merge"].eq("both").all():
        raise ValueError("SQL and Python record keys differ.")

    for column in ["player_aggregate_value", "event_export_value"]:
        python_values = pd.to_numeric(
            comparison[f"{column}_python"]
        ).to_numpy(dtype=float)

        sql_values = pd.to_numeric(
            comparison[f"{column}_sql"]
        ).to_numpy(dtype=float)

        if not np.isclose(
            python_values,
            sql_values,
            atol=1e-10,
            rtol=1e-10,
            equal_nan=True,
        ).all():
            raise ValueError(f"Values differ for {column}.")

    connection.commit()

    print("Advanced KPI import committed successfully.")
    print("SQL rows:", len(sql_advanced_check))
    print("Record keys match Python: YES")
    print("KPI values and missing values match Python: YES")

except Exception:
    connection.rollback()
    print("Import stopped; changes from this transaction rolled back.")
    raise

finally:
    connection.close()